# 01 — Exploration des données DVF

**Objectif :** premier coup d'œil sur les Demandes de Valeurs Foncières
récupérées via `src.data_loader` après l'étape de nettoyage.

**Sorties attendues :**
- compréhension du volume, de la couverture temporelle et géographique
- identification des valeurs aberrantes et patterns à corriger
- premières hypothèses sur les variables prédictives

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import CLEAN_PARQUET

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Chargement

In [ ]:
df = pd.read_parquet(CLEAN_PARQUET)
print(f"Shape : {df.shape}")
df.head()

In [ ]:
df.describe()

## Couverture temporelle

In [ ]:
df['date_mutation'] = pd.to_datetime(df['date_mutation'])
df_time = df.set_index('date_mutation').resample('M').size()

plt.figure(figsize=(12, 4))
df_time.plot()
plt.title("Nombre de transactions par mois")
plt.ylabel("Transactions")
plt.show()

## Distribution du prix au m²

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['prix_m2'], bins=80, ax=ax[0])
ax[0].set_title("Distribution prix/m²")
sns.histplot(df['prix_m2'].apply('log1p'), bins=80, ax=ax[1])
ax[1].set_title("Distribution log(prix/m²)")
plt.tight_layout()
plt.show()

La distribution est fortement asymétrique : la transformation logarithmique
stabilise la variance et justifie son emploi comme cible de modélisation.

## Prix par département

In [ ]:
top = (df.groupby('code_departement')['prix_m2']
       .median().sort_values(ascending=False).head(20))

plt.figure(figsize=(12, 4))
top.plot.bar()
plt.title("Top 20 départements (prix médian au m²)")
plt.ylabel("€/m²")
plt.show()